##1. Import Libraries

In [1]:
!pip install contractions

import pandas as pd
import re
import nltk
import spacy
import contractions

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

##2. Load Dataset

In [2]:
df = pd.read_csv("Tweets.csv")
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


##3. Explore Dataset

In [3]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing Values:")
print(df.isnull().sum())

Shape: (14640, 15)

Columns:
['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']

Missing Values:
tweet_id                            0
airline_sentiment                   0
airline_sentiment_confidence        0
negativereason                   5462
negativereason_confidence        4118
airline                             0
airline_sentiment_gold          14600
name                                0
negativereason_gold             14608
retweet_count                       0
text                                0
tweet_coord                     13621
tweet_created                       0
tweet_location                   4733
user_timezone                    4820
dtype: int64


##4. Select Text Data

In [4]:
data = df[['text', 'airline_sentiment']].copy()
data.head()

,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


##5. Display Original Text

In [5]:
for i in range(5):
    print("Original:", data['text'].iloc[i])
    print()

Original: @VirginAmerica What @dhepburn said.

Original: @VirginAmerica plus you've added commercials to the experience... tacky.

Original: @VirginAmerica I didn't today... Must mean I need to take another trip!

Original: @VirginAmerica it's really aggressive to blast obnoxious "entertainment" in your guests' faces &amp; they have little recourse

Original: @VirginAmerica and it's a really big bad thing about it



##6. Convert Text to Lowercase

In [6]:
data['clean_text'] = data['text'].str.lower()
data[['text', 'clean_text']].head()

,text,clean_text
0,@VirginAmerica What @dhepburn said.,@virginamerica what @dhepburn said.
1,@VirginAmerica plus you've added commercials t...,@virginamerica plus you've added commercials t...
2,@VirginAmerica I didn't today... Must mean I n...,@virginamerica i didn't today... must mean i n...
3,@VirginAmerica it's really aggressive to blast...,@virginamerica it's really aggressive to blast...
4,@VirginAmerica and it's a really big bad thing...,@virginamerica and it's a really big bad thing...


##7. Expand Contractions

In [7]:
data['clean_text'] = data['clean_text'].apply(contractions.fix)
data[['text', 'clean_text']].head()

,text,clean_text
0,@VirginAmerica What @dhepburn said.,@virginamerica what @dhepburn said.
1,@VirginAmerica plus you've added commercials t...,@virginamerica plus you have added commercials...
2,@VirginAmerica I didn't today... Must mean I n...,@virginamerica i did not today... must mean i ...
3,@VirginAmerica it's really aggressive to blast...,@virginamerica it is really aggressive to blas...
4,@VirginAmerica and it's a really big bad thing...,@virginamerica and it is a really big bad thin...


##8. Remove URLs

In [8]:
data['clean_text'] = data['clean_text'].apply(
    lambda x: re.sub(r'http\S+|www\S+|https\S+', '', x)
)
data[['text', 'clean_text']].head()

,text,clean_text
0,@VirginAmerica What @dhepburn said.,@virginamerica what @dhepburn said.
1,@VirginAmerica plus you've added commercials t...,@virginamerica plus you have added commercials...
2,@VirginAmerica I didn't today... Must mean I n...,@virginamerica i did not today... must mean i ...
3,@VirginAmerica it's really aggressive to blast...,@virginamerica it is really aggressive to blas...
4,@VirginAmerica and it's a really big bad thing...,@virginamerica and it is a really big bad thin...


##9. Remove Twitter Mentions

In [9]:
data['clean_text'] = data['clean_text'].apply(
    lambda x: re.sub(r'@\w+', '', x)
)
data[['text', 'clean_text']].head()

,text,clean_text
0,@VirginAmerica What @dhepburn said.,what said.
1,@VirginAmerica plus you've added commercials t...,plus you have added commercials to the experi...
2,@VirginAmerica I didn't today... Must mean I n...,i did not today... must mean i need to take a...
3,@VirginAmerica it's really aggressive to blast...,"it is really aggressive to blast obnoxious ""e..."
4,@VirginAmerica and it's a really big bad thing...,and it is a really big bad thing about it


##10. Remove Hashtags

In [10]:
data['clean_text'] = data['clean_text'].apply(
    lambda x: re.sub(r'#\w+', '', x)
)
data[['text', 'clean_text']].head()

,text,clean_text
0,@VirginAmerica What @dhepburn said.,what said.
1,@VirginAmerica plus you've added commercials t...,plus you have added commercials to the experi...
2,@VirginAmerica I didn't today... Must mean I n...,i did not today... must mean i need to take a...
3,@VirginAmerica it's really aggressive to blast...,"it is really aggressive to blast obnoxious ""e..."
4,@VirginAmerica and it's a really big bad thing...,and it is a really big bad thing about it


##11. Remove Unwanted Characters

In [11]:
data['clean_text'] = data['clean_text'].apply(
    lambda x: re.sub(r'[^a-zA-Z\s]', '', x)
)
data[['text', 'clean_text']].head()

,text,clean_text
0,@VirginAmerica What @dhepburn said.,what said
1,@VirginAmerica plus you've added commercials t...,plus you have added commercials to the experi...
2,@VirginAmerica I didn't today... Must mean I n...,i did not today must mean i need to take anot...
3,@VirginAmerica it's really aggressive to blast...,it is really aggressive to blast obnoxious en...
4,@VirginAmerica and it's a really big bad thing...,and it is a really big bad thing about it


##12. Remove Extra Whitespace

In [12]:
data['clean_text'] = data['clean_text'].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)
data[['text', 'clean_text']].head()

,text,clean_text
0,@VirginAmerica What @dhepburn said.,what said
1,@VirginAmerica plus you've added commercials t...,plus you have added commercials to the experie...
2,@VirginAmerica I didn't today... Must mean I n...,i did not today must mean i need to take anoth...
3,@VirginAmerica it's really aggressive to blast...,it is really aggressive to blast obnoxious ent...
4,@VirginAmerica and it's a really big bad thing...,and it is a really big bad thing about it


##13. Tokenization

In [13]:
nltk.download('punkt')
nltk.download('punkt_tab')

data['tokens'] = data['clean_text'].apply(word_tokenize)
data[['clean_text', 'tokens']].head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,clean_text,tokens
0,what said,"[what, said]"
1,plus you have added commercials to the experie...,"[plus, you, have, added, commercials, to, the,..."
2,i did not today must mean i need to take anoth...,"[i, did, not, today, must, mean, i, need, to, ..."
3,it is really aggressive to blast obnoxious ent...,"[it, is, really, aggressive, to, blast, obnoxi..."
4,and it is a really big bad thing about it,"[and, it, is, a, really, big, bad, thing, abou..."


##14. Download Stopwords

In [14]:
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


##15. Stop-word Removal

In [15]:
data['filtered_tokens'] = data['tokens'].apply(
    lambda tokens: [word for word in tokens if word not in stop_words]
)
data[['tokens', 'filtered_tokens']].head()

,tokens,filtered_tokens
0,"[what, said]",[said]
1,"[plus, you, have, added, commercials, to, the,...","[plus, added, commercials, experience, tacky]"
2,"[i, did, not, today, must, mean, i, need, to, ...","[today, must, mean, need, take, another, trip]"
3,"[it, is, really, aggressive, to, blast, obnoxi...","[really, aggressive, blast, obnoxious, enterta..."
4,"[and, it, is, a, really, big, bad, thing, abou...","[really, big, bad, thing]"


##16. Stemming

In [16]:
stemmer = PorterStemmer()

data['stemmed_tokens'] = data['filtered_tokens'].apply(
    lambda tokens: [stemmer.stem(word) for word in tokens]
)
data[['filtered_tokens', 'stemmed_tokens']].head()

,filtered_tokens,stemmed_tokens
0,[said],[said]
1,"[plus, added, commercials, experience, tacky]","[plu, ad, commerci, experi, tacki]"
2,"[today, must, mean, need, take, another, trip]","[today, must, mean, need, take, anoth, trip]"
3,"[really, aggressive, blast, obnoxious, enterta...","[realli, aggress, blast, obnoxi, entertain, gu..."
4,"[really, big, bad, thing]","[realli, big, bad, thing]"


##17. Load spaCy Model

In [17]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


##18. Lemmatization

In [18]:
nlp = spacy.load("en_core_web_sm")

def lemmatize_text(tokens):
    doc = nlp(" ".join(tokens))
    return [token.lemma_ for token in doc]

data['lemmatized_tokens'] = data['filtered_tokens'].apply(lemmatize_text)
data[['filtered_tokens', 'lemmatized_tokens']].head()

,filtered_tokens,lemmatized_tokens
0,[said],[say]
1,"[plus, added, commercials, experience, tacky]","[plus, add, commercial, experience, tacky]"
2,"[today, must, mean, need, take, another, trip]","[today, must, mean, need, take, another, trip]"
3,"[really, aggressive, blast, obnoxious, enterta...","[really, aggressive, blast, obnoxious, enterta..."
4,"[really, big, bad, thing]","[really, big, bad, thing]"


##19. Final Preprocessed Text

In [19]:
data['preprocessed_text'] = data['lemmatized_tokens'].apply(
    lambda tokens: ' '.join(tokens)
)
data[['text', 'preprocessed_text', 'airline_sentiment']].head(10)

,text,preprocessed_text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,say,neutral
1,@VirginAmerica plus you've added commercials t...,plus add commercial experience tacky,positive
2,@VirginAmerica I didn't today... Must mean I n...,today must mean need take another trip,neutral
3,@VirginAmerica it's really aggressive to blast...,really aggressive blast obnoxious entertainmen...,negative
4,@VirginAmerica and it's a really big bad thing...,really big bad thing,negative
5,@VirginAmerica seriously would pay $30 a fligh...,seriously would pay flight seat play really ba...,negative
6,"@VirginAmerica yes, nearly every time I fly VX...",yes nearly every time fly vx ear worm go away,positive
7,@VirginAmerica Really missed a prime opportuni...,really miss prime opportunity man without hat ...,neutral
8,"@virginamerica Well, I didn't…but NOW I DO! :-D",well notbut,positive
9,"@VirginAmerica it was amazing, and arrived an ...",amazing arrive hour early good,positive


##20. Compare Before and After Preprocessing

In [20]:
for i in range(5):
    print("BEFORE:")
    print(data['text'].iloc[i])
    print("\nAFTER:")
    print(data['preprocessed_text'].iloc[i])
    print("\n" + "="*80 + "\n")

BEFORE:
@VirginAmerica What @dhepburn said.

AFTER:
say


BEFORE:
@VirginAmerica plus you've added commercials to the experience... tacky.

AFTER:
plus add commercial experience tacky


BEFORE:
@VirginAmerica I didn't today... Must mean I need to take another trip!

AFTER:
today must mean need take another trip


BEFORE:
@VirginAmerica it's really aggressive to blast obnoxious "entertainment" in your guests' faces &amp; they have little recourse

AFTER:
really aggressive blast obnoxious entertainment guest face amp little recourse


BEFORE:
@VirginAmerica and it's a really big bad thing about it

AFTER:
really big bad thing




##21. Sentiment Distribution

In [21]:
print(data['airline_sentiment'].value_counts())

airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64


##22. Save Preprocessed Dataset

In [22]:
data.to_csv("Tweets_Preprocessed.csv", index=False)
print("Preprocessed dataset saved successfully.")

Preprocessed dataset saved successfully.
